# Chapter 9

## 추천 시스템

### 01. 추천시스템의 개요와 배경
추천 시스템은 크게 콘텐츠 기반 필터링 방식과 협업 필터링 방식으로 나뉜다. 


### 02. 콘텐츠 기반 필터링 추천 시스템 
콘텐츠 기반 필터링 : 사용자가 특정 아이템을 매우 선호하는 경우, 그 아이템과 비슷한 콘텐츠를 가진 다른 아이템을 추천하는 방식이다

### 03. 최근접 이웃 협업 필터링 
협업 필터링 :
- 구매 이력과 같은 사용자 행동 양식을 기반으로 추천
- 주요 목적 : 사용자-아이템 평점 매트릭스와 같은 축적된 사용자 행동 데이터를 기반으로 사용자가 아직 평가하지 않은 아이템을 예측 평가하는 것 
- 최근접 이웃 방식과 잠재 요인 방식

#### 최근접 이웃 방식 
- 메모리 협업 필터링이라고도 불린다
- 사용자 기반 : 당신과 비슷한 고객들이 다음 상품도 구매했습니다
- 아이템 기반 : 이 상품을 선택한 다른 고객들을 다음 상품도 구매했습니다. 
- 일반적으로 사용자 기반 보다 아이템 기반 협업 필터링이 정확도가 더 높다 
    - 왜냐면 비슷한 상품을 좋아한다고 해서 사람들의 취향이 비슷하다고 판단하기는 어려운 경우가 많기 때문이다
- 추천 시스템의 유사도 측정에는 코사인 유사도가 가장 많이 적용된다

### 04. 잠재 요인 협업 필터링 
잠재 요인 혐업 필터링 
- 사용자-아이템 평점 행렬 데이터만을 이용해 '잠재 요인'을 끄집어 내는 것 
- 잠재 요인이 어떤 것인지는 명확하게 정의할 수 없다

#### 행렬 분해의 이해
행렬 분해는 다차원의 매트릭스를 저차원 매트릭스로 분해하는 기법으로서 대표적으로 SVD, NMF 등이 있다. 
- M : 총 사용자 수 
- N : 총 아이템 수 
- K : 잠재 요인의 차원 수 
- R = M x N 차원의 사용자-아이템 평점 행렬
- P : 사용자와 잠재 요인과의 간계 값을 가지는 사용자-잠재요인 행렬
- Q : 아이템과 잠재 요인과의 관계 값을 가지는 차원의 아이템-잠재요인 행렬

행렬 분해는 주로 SVD 방법을 사용한다
- 하지만 SVD는 널 값이 없는 행렬에만 적용할 수 있다.
- 이러한 경우 확률적 경사 하강법 방식을 이용해 SVD 수행 


In [5]:
import numpy as np


# 원본 행렬 R 생성, 분해 행렬 P와 Q 초기화, 잠재 요인 차원 K는 3으로 설정.
R = np.array([[4, np.nan, np.nan, 2, np.nan],
              [np.nan, 5, np.nan, 3, 1],
              [np.nan, np.nan, 3, 4, 4],
              [5, 2, 1, 2, np.nan]])

num_users, num_items = R.shape
K=3


# P와 Q 행렬의 크기를 지정하고 정규 분포를 가진 임의의 값으로 입력합니다.
np.random.seed(1)

P = np.random.normal(scale=1./K, size=(num_users, K))

Q = np.random.normal(scale=1./K, size=(num_items, K))

In [6]:
from sklearn.metrics import mean_squared_error


def get_rmse(R, P, Q, non_zeros):
    error = 0
    # 두 개의 분해된 행렬 P와 Q.T의 내적으로 예측 R 행렬 생성
    full_pred_matrix = np.dot(P, Q.T)

    # 실제 R 행렬에서 널이 아닌 값의 위치 인덱스 추출해 실제 R 행렬과 예측 행렬의 RMSE 추출
    x_non_zero_ind = [non_zero[0] for non_zero in non_zeros]
    y_non_zero_ind = [non_zero[1] for non_zero in non_zeros]
    R_non_zeros = R[x_non_zero_ind, y_non_zero_ind]
    full_pred_matrix_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind]
    mse = mean_squared_error(R_non_zeros, full_pred_matrix_non_zeros)
    rmse = np.sqrt(mse)


    return rmse

In [7]:
# R > 0인 행 위치, 열 위치, 값을 non_zeros 리스트에 저장.
non_zeros = [ (i, j, R[i, j]) for i in range(num_users) for j in range(num_items) if R[i, j] > 0 ]


steps=1000

learning_rate=0.01

r_lambda=0.01


# SGD 기반으로 P와 Q 매트릭스를 계속 업데이트.
for step in range(steps):
    for i, j, r in non_zeros:
        # 실제 값과 예측 값의 차이인 오류 값 구함
        eij = r - np.dot(P[i, :], Q[j, :].T)
        # Regularization을 반영한 SGD 업데이트 공식 적용
        P[i, :] = P[i, :] + learning_rate*(eij * Q[j, :] - r_lambda*P[i, :])
        Q[j, :] = Q[j, :] + learning_rate*(eij * P[i, :] - r_lambda*Q[j, :])
        rmse = get_rmse(R, P, Q, non_zeros)
    if (step % 50) == 0 :
        print("### iteration step : ", step, " rmse : ", rmse)

### iteration step :  0  rmse :  3.2388050277987723
### iteration step :  50  rmse :  0.4876723101369648
### iteration step :  100  rmse :  0.15643403848192483
### iteration step :  150  rmse :  0.07455141311978061
### iteration step :  200  rmse :  0.04325226798579325
### iteration step :  250  rmse :  0.02924832878087924
### iteration step :  300  rmse :  0.022621116143829573
### iteration step :  350  rmse :  0.019493636196525155
### iteration step :  400  rmse :  0.018022719092132804
### iteration step :  450  rmse :  0.01731968595344285
### iteration step :  500  rmse :  0.01697365788757083
### iteration step :  550  rmse :  0.016796804595895533
### iteration step :  600  rmse :  0.016701322901884485
### iteration step :  650  rmse :  0.01664473691247668
### iteration step :  700  rmse :  0.01660591006820996
### iteration step :  750  rmse :  0.016574200475704987
### iteration step :  800  rmse :  0.016544315829216078
### iteration step :  850  rmse :  0.01651375177473504
### iter

In [8]:
pred_matrix = np.dot(P, Q.T)
print('예측 행렬:\n', np.round(pred_matrix, 3))

예측 행렬:
 [[3.991 0.897 1.306 2.002 1.663]
 [6.696 4.978 0.979 2.981 1.003]
 [6.677 0.391 2.987 3.977 3.986]
 [4.968 2.005 1.006 2.017 1.14 ]]
